In [1]:
from openai import OpenAI
import json
from tqdm import tqdm
from api_keys.openai import openai_api_key
client = OpenAI(
    api_key=openai_api_key
)

system_prompt = """
You're an expert in scanning probe microscopy (SPM). I will give you a technical question related to SPM along with reference material. Then, I will provide an original answer to the question, which is likely non-expert and may contain inaccuracies. Your task is to revise the original answer to make it more accurate, complete, and aligned with the reference.
"""



def distill_answer(question, reference, answer, model: str = "o4-mini") -> str:
    try:
        # Prompt caching lets OpenAI recognize repeated input patterns and reuse prior completions or avoid recomputation. For the case that Reference is the same every time.
        completion = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Reference:\n{reference}"},
                {"role": "user", "content": f"Question: {question}"},
                {"role": "user", "content": f"Original Answer: {answer}"},
                {"role": "user", "content": "Improved Answer:"}
            ],
            reasoning_effort="high",
        )

        return completion.choices[0].message.content
    except Exception as e:
        print(f"Error distilling answer: {e}")
        return ""


def distill_answer_to_dataset(path):
    with open(path, "r") as json_file:
        data = json.load(json_file)
    if data["type"] == "journal" and data["distilled"] == False:
        __distill_answer_to_journal_dataset(path)
    elif data["type"] == "book" and data["distilled"] == False:
        __distill_answer_to_book_dataset(path)


def __distill_answer_to_book_dataset(path):
    with open(path, "r") as json_file:
        data = json.load(json_file)

    if data["type"] == "book" and data["answered"] == True and data["distilled"] == False:
        texts = data.pop("text")

        for chapter_idx, text in enumerate(tqdm(texts)):
            distill_answer_list = []
            for i, it in enumerate(data[str(chapter_idx)]):
                print(f"{chapter_idx}-{i}")
                print("question: {}".format(data[str(chapter_idx)][i]))
                print("answer: {}".format(data[f"{chapter_idx}_answers"][i]))
                d_answer = distill_answer(question=data[str(chapter_idx)][i],
                                          answer=data[f"{chapter_idx}_answers"][i],
                                          reference=texts[chapter_idx])
                distill_answer_list.append(d_answer)
                print("distill_answer: {}".format(d_answer))
            
            data[f"{chapter_idx}_distilled_answers"] = distill_answer_list

        data["answered"] = True
        data["distilled"] = True
        data["text"] = texts
        with open(path, "w") as json_file:
            json.dump(data, json_file, indent=4)
            

def __distill_answer_to_journal_dataset(path):
    with open(path, "r") as json_file:
        data = json.load(json_file)
    
    distill_answer_list = []
    if data["type"] == "journal" and data["answered"] == True and data["distilled"] == False:
        for i,it in enumerate(tqdm(data["questions"])):
            print("question:", it)
            print("answer:", data["answers"][i])
            d_answer = distill_answer(question=it, answer=data["answers"][i], reference=data["text"])
            distill_answer_list.append(d_answer)
            print("distill_answer: {}".format(d_answer))

    data["distilled_answers"] = distill_answer_list
    data["distilled"] = True
    with open(path, "w") as json_file:
        json.dump(data, json_file, indent=4)


In [2]:
distill_answer_to_dataset(
    "./outputs/Small Methods - 2024 - Diao - AI‐Equipped Scanning Probe Microscopy for Autonomous Site‐Specific Atomic‐Level.pdf.json")

  0%|          | 0/39 [00:00<?, ?it/s]

question: What is the primary purpose of integrating AI into scanning probe microscopy (SPM) systems?
answer: The primary purpose of integrating artificial intelligence (AI) into scanning probe microscopy (SPM) systems is to enhance the precision, efficiency, and reliability of atomic-scale measurements and surface analysis. AI-equipped SPM systems, such as the AI-SPM described in the article, are designed to autonomously identify and manipulate atomic positions with high precision. This integration allows for several key advancements:

1. **Autonomous Operation**: AI enables the SPM system to perform complex tasks such as spectroscopic data acquisition and atomic adjustment without constant human intervention. This includes the ability to detect and adapt to surface defects, targeting or avoiding them as necessary.

2. **Overcoming Challenges**: AI helps overcome typical challenges associated with SPM, such as positional drift and tip apex atomic variations due to thermal effects. Thi

  3%|▎         | 1/39 [00:19<12:07, 19.13s/it]

distill_answer: The principal aim of embedding AI into an SPM is to turn what has traditionally been a manually-driven, expert-dependent instrument into a self-driving platform for atomic-scale surface analysis.  In practice this means:

1. Real-time Situational Assessment  
   – CNN “Net1” classifies tip‐apex and surface quality (good vs. contaminated, capable of atomic resolution or not).  
   – “Net2” locates immobile and mobile adsorbates.  
   – “Net3” identifies half-unit cells and pinpoints key atomic sites (corner holes, corner adatoms, center adatoms).

2. Automated Environmental Control  
   – A drift-feedforward module continually measures and compensates thermal drift via feature‐point matching.  
   – A tip-conditioning routine automatically “pokes” and reshapes the tip apex until Net1 reports an optimal state.

3. Self-Driving, Site-Specific Measurement  
   – Using the AI‐inferred map of defects, adsorbates, and atomic sites, the system autonomously navigates to defect-f

  5%|▌         | 2/39 [00:37<11:25, 18.52s/it]

...

distill_answer: In the AI-SPM’s post-processing of large sets of room-temperature I–V curves, the cosine-similarity metric is used to isolate the most internally consistent data and suppress outliers caused by tip changes, thermal drift or electronic noise.  Its implementation proceeds in three steps:

1. Pairwise similarity calculation  
   – For every pair of I–V curves I_i and I_j (each sampled at n bias points), the system computes  
     φ(I_i,I_j) = ∑ₖ[I_i(k)·I_j(k)] / (‖I_i‖·‖I_j‖)  
     where φ ranges from –1 (anti-correlated) to +1 (identical).  

2. Group selection  
   • Overall selection mode:  
     – Compute φ_mean(I_l) = (1/M)∑_{i=1…M}φ(I_l,I_i) for each curve I_l in the full set of M curves.  
     – Retain only those I_l for which φ_mean(I_l) exceeds a user-defined threshold T₁.  
   • Reference-based mode:  
     – Choose a single “reference” I–V curve I_ref.  
     – Retain all curves I_l for which φ(I_ref,I_l) > T₂.  

   By either criterion, the AI-SPM automatical

 97%|█████████▋| 38/39 [07:50<00:13, 13.78s/it]

distill_answer: The AI-SPM guarantees both accuracy and reproducibility by closing the loop between real-time AI inference, active instrument control, and statistical data validation:

1. Composite CNN-Based Quality Assessment  
   • Three networks (Net1–Net3) jointly evaluate every STM frame:  
     – Net1 classifies tip‐apex and surface quality into 11 states (K0–K10) with 87–90 % accuracy.  
     – Net2 detects and distinguishes stationary and mobile adsorbates (≈92 % accuracy).  
     – Net3 localizes half‐unit cells and nine key adatom/corner-hole sites (≈91 % accuracy).  
   • These models were trained on >40 000 in situ images acquired via a two-phase routine (Phase 1: broad coverage of tip states and drift conditions; Phase 2: high-resolution scans via automated atom manipulation) and augmented to ensure robustness to tip variations and surface defects.

2. Real-Time Thermal Drift Compensation  
   • A feed-forward feature‐matching algorithm tracks and matches corners and defec

100%|██████████| 39/39 [08:25<00:00, 12.96s/it]

distill_answer: Integrating AI into SPM fundamentally transforms it from a manually steered microscope into a self-driving, high-throughput platform.  The key benefits demonstrated in the AI-SPM work on Si(111)-(7×7) are:

1. Autonomous Tip-Health and Drift Management  
   • A CNN (Net1) continuously classifies tip-apex and surface conditions (good/contaminated) with ≈90 % accuracy.  
   • When the tip degrades, the system automatically performs controlled “pokes” (bias/current pulses plus approach) until atomic resolution is restored.  
   • A feature-point–matching feed-forward algorithm tracks thermal drift on minute-to-day timescales and adjusts the scan area in real time, keeping tip and sample atoms locked together at room temperature.

2. Real-Time, Site-Specific Decision Making  
   • A second CNN (Net2) discriminates immobile versus mobile adsorbates, while a third (Net3) locates half-unit cells (faulted vs. unfaulted) and nine key adatom/corner-hole positions.  
   • This let